In [ ]:
!nvidia-smi

Wed Oct  7 04:33:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:

import os
import time
import subprocess
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("GPU unavailable. Select a GPU runtime in Colab.")

print("GPU:", torch.cuda.get_device_name(0))
print("GPU capability:", torch.cuda.get_device_capability(0))

result = subprocess.run(
    ["nvidia-smi"],
    capture_output=True,
    text=True
)
print(result.stdout)


PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU capability: (7, 5)
Sun Oct 11 06:08:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                 

In [ ]:

%pip install -q "transformers>=4.50" accelerate pandas


In [ ]:

import subprocess
import json
import time
import torch
import pandas as pd

def read_gpu():
    fields = [
        "utilization.gpu",
        "utilization.memory",
        "memory.used",
        "memory.total",
        "power.draw",
        "temperature.gpu",
    ]

    result = subprocess.run(
        [
            "nvidia-smi",
            f"--query-gpu={','.join(fields)}",
            "--format=csv,noheader,nounits",
        ],
        capture_output=True,
        text=True,
        check=True,
    )

    values = [v.strip() for v in result.stdout.strip().split(",")]

    return {
        "gpu_util_pct": int(values[0]),
        "memory_util_pct": int(values[1]),
        "memory_used_mib": int(values[2]),
        "memory_total_mib": int(values[3]),
        "power_w": float(values[4]) if values[4] not in ("[N/A]", "N/A") else None,
        "temperature_c": int(values[5]),
    }

idle_samples = []

for i in range(10):
    sample = {"sample": i, "timestamp": time.time(), **read_gpu()}
    idle_samples.append(sample)
    time.sleep(1)

idle_df = pd.DataFrame(idle_samples)
display(idle_df)

print("\nIdle summary:")
display(idle_df.drop(columns=["sample", "timestamp"]).mean().to_frame("mean"))


,sample,timestamp,gpu_util_pct,memory_util_pct,memory_used_mib,memory_total_mib,power_w,temperature_c
0,0,1.791699e+09,0,0,3,15360,13.13,37
1,1,1.791699e+09,0,0,3,15360,13.23,37
2,2,1.791699e+09,0,0,3,15360,13.13,37
3,3,1.791699e+09,0,0,3,15360,13.23,37
4,4,1.791699e+09,0,0,3,15360,13.23,38
5,5,1.791699e+09,0,0,3,15360,13.13,38
6,6,1.791699e+09,0,0,3,15360,13.13,38
7,7,1.791699e+09,0,0,3,15360,13.13,38
8,8,1.791699e+09,0,0,3,15360,13.14,38
9,9,1.791699e+09,0,0,3,15360,13.13,38



Idle summary:


,mean
gpu_util_pct,0.000
memory_util_pct,0.000
memory_used_mib,3.000
memory_total_mib,15360.000
power_w,13.161
temperature_c,37.600


In [ ]:

import time
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Load time is measured separately.
torch.cuda.reset_peak_memory_stats()
load_start = time.perf_counter()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda",
)
model.eval()

torch.cuda.synchronize()
load_seconds = time.perf_counter() - load_start

print(f"Model load time: {load_seconds:.2f} s")
print(f"Model GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
print(f"Model peak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")

prompt = (
    "Explain how an LLM generates the next token. "
    "Describe prefill, attention, and autoregressive decoding."
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
print("Input tokens:", inputs["input_ids"].shape[1])

# Warm-up: excluded from the measured runs.
with torch.inference_mode():
    _ = model.generate(
        **inputs,
        max_new_tokens=16,
        do_sample=False,
        use_cache=True,
    )
torch.cuda.synchronize()

# Measure several runs; synchronize before and after each request.
runs = []

for run_id in range(5):
    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=False,
            use_cache=True,
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    prompt_tokens = inputs["input_ids"].shape[1]
    total_tokens = output.shape[1]
    output_tokens = total_tokens - prompt_tokens

    runs.append({
        "run": run_id + 1,
        "latency_s": elapsed,
        "prompt_tokens": prompt_tokens,
        "output_tokens": output_tokens,
        "output_tok_s": output_tokens / elapsed,
    })

runs_df = pd.DataFrame(runs)
display(runs_df)

print(f"Mean latency: {runs_df['latency_s'].mean():.3f} s")
print(f"Mean output throughput: {runs_df['output_tok_s'].mean():.2f} tokens/s")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


... (3 outputs left out)

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model load time: 45.16 s
Model GPU memory allocated: 2.88 GiB
Model peak GPU memory: 2.88 GiB
Input tokens: 23


,run,latency_s,prompt_tokens,output_tokens,output_tok_s
0,1,3.472857,23,100,28.794739
1,2,3.575774,23,100,27.965970
2,3,3.971230,23,100,25.181118
3,4,3.645657,23,100,27.429898
4,5,3.490536,23,100,28.648892


Mean latency: 3.631 s
Mean output throughput: 27.60 tokens/s
Peak GPU memory: 2.89 GiB


In [ ]:

import time
import threading
import subprocess
import pandas as pd
import torch

gpu_samples = []
request_results = []
stop_monitor = threading.Event()

def sample_gpu():
    fields = [
        "utilization.gpu",
        "utilization.memory",
        "memory.used",
        "power.draw",
        "temperature.gpu",
    ]
    while not stop_monitor.is_set():
        try:
            result = subprocess.run(
                [
                    "nvidia-smi",
                    f"--query-gpu={','.join(fields)}",
                    "--format=csv,noheader,nounits",
                ],
                capture_output=True,
                text=True,
                check=True,
            )
            v = [x.strip() for x in result.stdout.strip().split(",")]
            gpu_samples.append({
                "elapsed_s": time.perf_counter() - monitor_start,
                "gpu_util_pct": int(v[0]),
                "memory_util_pct": int(v[1]),
                "memory_used_mib": int(v[2]),
                "power_w": float(v[3]) if v[3] not in ("N/A", "[N/A]") else None,
                "temperature_c": int(v[4]),
            })
        except Exception as e:
            print("GPU sample error:", e)
        stop_monitor.wait(0.5)

def run_workload():
    for i in range(8):
        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            output = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                use_cache=True,
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        generated = output.shape[1] - inputs["input_ids"].shape[1]

        request_results.append({
            "request": i + 1,
            "latency_s": elapsed,
            "output_tokens": generated,
            "output_tok_s": generated / elapsed,
        })

monitor_start = time.perf_counter()
monitor_thread = threading.Thread(target=sample_gpu)
monitor_thread.start()

try:
    run_workload()
finally:
    stop_monitor.set()
    monitor_thread.join()

samples_df = pd.DataFrame(gpu_samples)
requests_df = pd.DataFrame(request_results)

print("Request results")
display(requests_df)

print("GPU activity summary")
display(samples_df[[
    "gpu_util_pct", "memory_util_pct", "memory_used_mib",
    "power_w", "temperature_c"
]].agg(["mean", "median", "max", "min"]).round(2))

print("GPU samples collected:", len(samples_df))


Request results


,request,latency_s,output_tokens,output_tok_s
0,1,18.798296,200,10.639262
1,2,15.290109,200,13.080352
2,3,14.109769,200,14.174576
3,4,7.789385,200,25.675970
4,5,7.124299,200,28.072939
5,6,7.790374,200,25.672707
6,7,7.682966,200,26.031613
7,8,7.169863,200,27.894536


GPU activity summary


,gpu_util_pct,memory_util_pct,memory_used_mib,power_w,temperature_c
mean,38.34,31.93,3143.0,48.03,62.71
median,41.00,31.00,3143.0,40.02,59.00
max,56.00,49.00,3143.0,72.83,75.00
min,0.00,0.00,3143.0,28.55,54.00


GPU samples collected: 153


In [ ]:

import time
import torch
import pandas as pd

# Additional warm-up; excluded from results.
with torch.inference_mode():
    for _ in range(3):
        _ = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            use_cache=True,
        )
torch.cuda.synchronize()

steady_results = []

for i in range(10):
    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            use_cache=True,
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    generated = output.shape[1] - inputs["input_ids"].shape[1]

    steady_results.append({
        "request": i + 1,
        "latency_s": elapsed,
        "output_tokens": generated,
        "output_tok_s": generated / elapsed,
    })

steady_df = pd.DataFrame(steady_results)
display(steady_df)

print("\nSteady-state summary")
display(
    steady_df[["latency_s", "output_tok_s"]]
    .agg(["mean", "median", "std", "min", "max"])
    .round(3)
)


,request,latency_s,output_tokens,output_tok_s
0,1,7.590964,200,26.347114
1,2,6.830390,200,29.280907
2,3,7.573531,200,26.407761
3,4,7.023838,200,28.474461
4,5,7.358464,200,27.179585
5,6,7.602157,200,26.308324
6,7,6.960476,200,28.733667
7,8,7.587440,200,26.359353
8,9,7.037226,200,28.420288
9,10,7.584865,200,26.368300



Steady-state summary


,latency_s,output_tok_s
mean,7.315,27.388
median,7.466,26.794
std,0.316,1.201
min,6.830,26.308
max,7.602,29.281


In [ ]:

import time
import torch
import pandas as pd

# Repeat the same prompt to create batches of different sizes.
prompt = "Explain how an LLM generates the next token."
encoded = tokenizer(prompt, return_tensors="pt")
single_ids = encoded["input_ids"].to("cuda")
single_mask = encoded["attention_mask"].to("cuda")

batch_results = []

for batch_size in [1, 2, 4, 8]:
    input_ids = single_ids.repeat(batch_size, 1)
    attention_mask = single_mask.repeat(batch_size, 1)

    # Warm-up for this batch size.
    with torch.inference_mode():
        _ = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=50,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()

    latencies = []

    for _ in range(3):
        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            output = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=50,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

        torch.cuda.synchronize()
        latencies.append(time.perf_counter() - start)

    mean_latency = sum(latencies) / len(latencies)
    total_output_tokens = batch_size * 50

    batch_results.append({
        "batch_size": batch_size,
        "mean_batch_latency_s": mean_latency,
        "aggregate_output_tok_s": total_output_tokens / mean_latency,
        "approx_per_sequence_tok_s": 50 / mean_latency,
        "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
    })

batch_df = pd.DataFrame(batch_results)
display(batch_df.round(3))


,batch_size,mean_batch_latency_s,aggregate_output_tok_s,approx_per_sequence_tok_s,peak_allocated_gib
0,1,4.075,12.269,12.269,2.896
1,2,3.066,32.617,16.308,2.896
2,4,1.904,105.060,26.265,2.901
3,8,1.939,206.318,25.790,2.918


In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

PyTorch: 2.11.0+cu130
Transformers: 5.18.0
CUDA available: True
GPU: Tesla T4
CUDA: 13.0


In [ ]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 107.8 MB/s eta 0:00:00


In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cuda"
)

model.eval()

print("Device:", model.device)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e9, "B")

In [ ]:
print(model.config)

In [ ]:
prompt = "Explain how an LLM generates the next token."

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Input tokens:", inputs["input_ids"].shape[1])

Input tokens: 11


In [ ]:
with torch.inference_mode():
    _ = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False
    )

torch.cuda.synchronize()

In [ ]:
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()

start = time.perf_counter()

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False
    )

torch.cuda.synchronize()

elapsed = time.perf_counter() - start

input_tokens = inputs["input_ids"].shape[1]
output_tokens = output.shape[1] - input_tokens

peak_memory = torch.cuda.max_memory_allocated() / 1024**3
tok_per_sec = output_tokens / elapsed

print(f"Input tokens:       {input_tokens}")
print(f"Output tokens:      {output_tokens}")
print(f"Latency:            {elapsed:.3f} s")
print(f"Output tok/s:       {tok_per_sec:.2f}")
print(f"Peak GPU memory:    {peak_memory:.3f} GB")

Input tokens:       11
Output tokens:      100
Latency:            5.469 s
Output tok/s:       18.29
Peak GPU memory:    2.891 GB


In [ ]:
print(tokenizer.decode(output[0], skip_special_tokens=True))

Explain how an LLM generates the next token. An LLM (Language Model) is a type of artificial intelligence that can generate human-like text based on its training data. The process by which an LLM generates the next token in a sequence of words or characters is called "decoding" and it involves several steps:

1. **Input**: The input to the LLM is typically a sequence of tokens, such as individual letters, words, or sentences.

2. **Contextual Information**: During training, LLMs are exposed to vast


In [ ]:
# Prefill measurement
torch.cuda.synchronize()

start = time.perf_counter()

with torch.inference_mode():
    outputs = model(
        **inputs,
        use_cache=True
    )

torch.cuda.synchronize()

prefill_time = time.perf_counter() - start

print(f"Prefill time: {prefill_time:.6f} s")
print(f"Logits shape: {outputs.logits.shape}")
print(f"KV cache layers: {len(outputs.past_key_values)}")

Prefill time: 0.137490 s
Logits shape: torch.Size([1, 11, 151936])
KV cache layers: 28


In [ ]:
base_text = """
Explain how large language models work, including tokenization,
embeddings, transformer layers, attention, KV cache, logits,
sampling, and autoregressive generation.
"""

for target_tokens in [500, 1000, 2000]:
    text = base_text * ((target_tokens // 45) + 1)

    test_inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=target_tokens
    ).to("cuda")

    actual_input_tokens = test_inputs["input_ids"].shape[1]

    # Warmup
    with torch.inference_mode():
        _ = model.generate(
            **test_inputs,
            max_new_tokens=10,
            do_sample=False
        )

    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **test_inputs,
            max_new_tokens=100,
            do_sample=False
        )

    torch.cuda.synchronize()

    elapsed = time.perf_counter() - start
    output_tokens = output.shape[1] - actual_input_tokens
    peak_memory = torch.cuda.max_memory_allocated() / 1024**3

    print(
        f"Input: {actual_input_tokens:4d} | "
        f"Latency: {elapsed:.3f}s | "
        f"Output tok/s: {output_tokens / elapsed:.2f} | "
        f"Peak GPU: {peak_memory:.3f} GB"
    )

Input:  397 | Latency: 1.575s | Output tok/s: 17.15 | Peak GPU: 2.929 GB
Input:  760 | Latency: 1.697s | Output tok/s: 15.91 | Peak GPU: 2.996 GB
Input: 1486 | Latency: 1.618s | Output tok/s: 16.68 | Peak GPU: 3.214 GB


In [ ]:
!pip install -q vllm==0.30.0 requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 314.9/314.9 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 88.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 125.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 94.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2/43.2 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 729

In [ ]:
!pip uninstall -y torchaudio

Found existing installation: torchaudio 2.11.0+cu130
Uninstalling torchaudio-2.11.0+cu130:
  Successfully uninstalled torchaudio-2.11.0+cu130


In [ ]:
!pkill -f "vllm serve" || true

!nohup vllm serve Qwen/Qwen2.5-1.5B-Instruct \
  --dtype float16 \
  --gpu-memory-utilization 0.8 \
  --max-model-len 4096 \
  --port 8000 \
  > /tmp/vllm.log 2>&1 &

^C


In [ ]:
!tail -30 /tmp/vllm.log

(APIServer pid=2398) INFO 10-08 08:01:52 [entry.py:139] Starting vLLM server on http://0.0.0.0:8000
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:61] Available routes are:
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /openapi.json, Methods: HEAD, GET
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /docs, Methods: HEAD, GET
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /docs/oauth2-redirect, Methods: HEAD, GET
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /redoc, Methods: HEAD, GET
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /load, Methods: GET
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /version, Methods: GET
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /health, Methods: GET
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /metrics, Methods: GET
(APIServer pid=2398) INFO 10-08 08:01:52 [launcher.py:70] Route: /tokenize, Methods

In [ ]:
import csv
import json
import re
import time
import threading
import requests

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


# ============================================================
# Configuration
# ============================================================

BASE_URL = "http://127.0.0.1:8000"
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

# Large single-shot workload.
CONCURRENCY = 128

# Leave enough context room for generation.
TARGET_PROMPT_TOKENS = 3000
MAX_NEW_TOKENS = 100

# High-frequency scheduler sampling.
SAMPLE_INTERVAL = 0.05

OUTPUT_DIR = Path("/content/phase6_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# vLLM metrics
# ============================================================

def get_metrics():
    response = requests.get(
        f"{BASE_URL}/metrics",
        timeout=5,
    )
    response.raise_for_status()

    text = response.text

    def gauge(pattern, default=0.0):
        match = re.search(pattern, text, re.MULTILINE)
        return float(match.group(1)) if match else default

    return {
        "running": gauge(
            r'^vllm:num_requests_running\s+([0-9.eE+-]+)',
            0,
        ),

        "waiting": gauge(
            r'^vllm:num_requests_waiting\s+([0-9.eE+-]+)',
            0,
        ),

        "kv_cache_usage": gauge(
            r'^vllm:kv_cache_usage_perc\s+([0-9.eE+-]+)',
            0,
        ),

        "capacity_waiting": gauge(
            r'^vllm:num_requests_waiting_by_reason\{reason="capacity"\}\s+([0-9.eE+-]+)',
            0,
        ),

        "deferred_waiting": gauge(
            r'^vllm:num_requests_waiting_by_reason\{reason="deferred"\}\s+([0-9.eE+-]+)',
            0,
        ),
    }


# ============================================================
# Prompt construction
# ============================================================

unit = """
Large language model inference systems must manage GPU compute,
memory bandwidth, request scheduling, and KV cache capacity.

During serving, multiple requests compete for finite GPU resources.
The scheduler decides which requests can execute while respecting
available KV cache and execution budgets.

Autoregressive generation maintains a key-value cache for every
transformer layer. As the context grows, each active request consumes
more KV cache memory.

With many concurrent requests, the aggregate KV-cache requirement can
become larger than the available memory reserved for inference.

A production inference engine therefore cannot simply execute every
incoming request at the same time. It continuously decides which
requests can remain in the running set and which requests must wait
for resources to become available.

This creates an important scheduling problem. The inference engine
must balance GPU computation, memory capacity, request latency,
throughput, and fairness between active requests.

Long prompts require substantial prefill computation and KV-cache
allocation. During decoding, every generated token extends the
request's KV cache.

When many long-context requests arrive together, the scheduler may
need to keep some requests waiting until sufficient KV-cache blocks
become available.

The goal of this experiment is to observe this behavior directly
under a large concurrent workload.
"""


# ============================================================
# Tokenize prompt through vLLM
# ============================================================

def tokenize_prompt(text):
    """
    Use vLLM's tokenizer endpoint so the experiment knows
    the actual token count rather than estimating from characters.
    """

    response = requests.post(
        f"{BASE_URL}/tokenize",
        json={
            "model": MODEL,
            "prompt": text,
        },
        timeout=30,
    )

    response.raise_for_status()

    data = response.json()

    tokens = data.get("tokens", [])

    return tokens


# Build a sufficiently large candidate prompt.
candidate_prompt = unit * 200

print("Checking vLLM server...")

initial_metrics = get_metrics()

print(initial_metrics)

print()
print("Tokenizing workload...")


candidate_tokens = tokenize_prompt(candidate_prompt)

print(f"Candidate tokens: {len(candidate_tokens):,}")


# If the candidate is smaller than the target, repeat it.
while len(candidate_tokens) < TARGET_PROMPT_TOKENS:

    candidate_prompt += unit
    candidate_tokens = tokenize_prompt(candidate_prompt)


# We deliberately stay below max-model-len=4096.
# Target ~3000 input tokens + 100 output tokens.
prompt_tokens = candidate_tokens[:TARGET_PROMPT_TOKENS]


# Decode the selected tokens back into text using the tokenizer API.
# vLLM's detokenize endpoint lets us preserve an exact token target.
response = requests.post(
    f"{BASE_URL}/detokenize",
    json={
        "model": MODEL,
        "tokens": prompt_tokens,
    },
    timeout=30,
)

response.raise_for_status()

prompt = response.json()["prompt"]

# Verify final token count.
final_tokens = tokenize_prompt(prompt)

print(f"Final prompt tokens: {len(final_tokens):,}")
print(f"Prompt characters:    {len(prompt):,}")

if len(final_tokens) + MAX_NEW_TOKENS > 4096:
    raise RuntimeError(
        f"Prompt too large: {len(final_tokens)} input + "
        f"{MAX_NEW_TOKENS} output > 4096 max-model-len"
    )


# ============================================================
# Request worker
# ============================================================

def request_one(request_id):

    start = time.perf_counter()

    try:

        response = requests.post(
            f"{BASE_URL}/v1/completions",
            json={
                "model": MODEL,
                "prompt": prompt,
                "max_tokens": MAX_NEW_TOKENS,
                "temperature": 0,
            },
            timeout=300,
        )

        latency = time.perf_counter() - start

        if not response.ok:

            return {
                "id": request_id,
                "success": False,
                "status_code": response.status_code,
                "error": response.text[:2000],
                "latency": latency,
                "prompt_tokens": 0,
                "completion_tokens": 0,
            }

        data = response.json()

        usage = data.get("usage", {})

        return {
            "id": request_id,
            "success": True,
            "status_code": response.status_code,
            "error": None,
            "latency": latency,
            "prompt_tokens": usage.get(
                "prompt_tokens",
                len(final_tokens),
            ),
            "completion_tokens": usage.get(
                "completion_tokens",
                0,
            ),
        }

    except Exception as e:

        return {
            "id": request_id,
            "success": False,
            "status_code": None,
            "error": str(e),
            "latency": time.perf_counter() - start,
            "prompt_tokens": 0,
            "completion_tokens": 0,
        }


# ============================================================
# Background metrics sampler
# ============================================================

samples = []

stop_sampling = False


def sampler():

    while not stop_sampling:

        try:

            sample = get_metrics()

            sample["time"] = time.perf_counter()

            samples.append(sample)

        except Exception:
            pass

        time.sleep(SAMPLE_INTERVAL)


# ============================================================
# Percentile
# ============================================================

def percentile(values, p):

    if not values:
        return 0.0

    values = sorted(values)

    k = (len(values) - 1) * p / 100

    lower = int(k)
    upper = min(lower + 1, len(values) - 1)

    return (
        values[lower]
        + (values[upper] - values[lower])
        * (k - lower)
    )


# ============================================================
# Warm-up
# ============================================================

print()
print("Running warm-up...")

warmup = requests.post(
    f"{BASE_URL}/v1/completions",
    json={
        "model": MODEL,
        "prompt": "Explain KV cache in one sentence.",
        "max_tokens": 10,
        "temperature": 0,
    },
    timeout=120,
)

if not warmup.ok:
    print("Warm-up failed:")
    print(warmup.text)
    raise RuntimeError("vLLM warm-up request failed.")


print("Warm-up complete.")


# ============================================================
# Experiment
# ============================================================

print()
print("========================================")
print("PHASE 6 RESOURCE PRESSURE EXPERIMENT")
print("========================================")

print(f"Model:                {MODEL}")
print(f"Concurrency:          {CONCURRENCY}")
print(f"Prompt tokens:        {len(final_tokens):,}")
print(f"Max new tokens:       {MAX_NEW_TOKENS}")
print(f"Total token/request:  {len(final_tokens) + MAX_NEW_TOKENS:,}")
print(f"Sampling interval:    {SAMPLE_INTERVAL}s")

print()
print("Starting workload...")
print()


samples.clear()
stop_sampling = False

sampler_thread = threading.Thread(
    target=sampler,
    daemon=True,
)

sampler_thread.start()


wall_start = time.perf_counter()

results = []


with ThreadPoolExecutor(
    max_workers=CONCURRENCY
) as executor:

    futures = [
        executor.submit(
            request_one,
            request_id,
        )
        for request_id in range(CONCURRENCY)
    ]

    for future in as_completed(futures):

        result = future.result()

        results.append(result)

        # Lightweight progress indicator.
        if len(results) % 16 == 0:

            print(
                f"Completed: {len(results)}/{CONCURRENCY}"
            )


wall_time = time.perf_counter() - wall_start


stop_sampling = True

sampler_thread.join(timeout=2)


# Final sample
try:

    final_sample = get_metrics()

    final_sample["time"] = time.perf_counter()

    samples.append(final_sample)

except Exception:
    pass


# ============================================================
# Separate successful / failed requests
# ============================================================

successful = [
    result
    for result in results
    if result["success"]
]

failed = [
    result
    for result in results
    if not result["success"]
]


# ============================================================
# Aggregate measurements
# ============================================================

completed = len(successful)
failed_count = len(failed)

total_prompt_tokens = sum(
    result["prompt_tokens"]
    for result in successful
)

total_output_tokens = sum(
    result["completion_tokens"]
    for result in successful
)

latencies = [
    result["latency"]
    for result in successful
]


mean_latency = (
    sum(latencies) / len(latencies)
    if latencies
    else 0
)

p50_latency = percentile(
    latencies,
    50,
)

p95_latency = percentile(
    latencies,
    95,
)

p99_latency = percentile(
    latencies,
    99,
)

request_throughput = (
    completed / wall_time
    if wall_time > 0
    else 0
)

output_tok_s = (
    total_output_tokens / wall_time
    if wall_time > 0
    else 0
)


# ============================================================
# Peak scheduler metrics
# ============================================================

max_running = max(
    (
        sample["running"]
        for sample in samples
    ),
    default=0,
)

max_waiting = max(
    (
        sample["waiting"]
        for sample in samples
    ),
    default=0,
)

max_capacity_waiting = max(
    (
        sample["capacity_waiting"]
        for sample in samples
    ),
    default=0,
)

max_deferred_waiting = max(
    (
        sample["deferred_waiting"]
        for sample in samples
    ),
    default=0,
)

peak_kv = max(
    (
        sample["kv_cache_usage"]
        for sample in samples
    ),
    default=0,
)


# ============================================================
# Print final results
# ============================================================

print()
print("========================================")
print("RESULTS")
print("========================================")

print()
print("Requests:")
print(
    f"Successful requests:       "
    f"{completed}/{CONCURRENCY}"
)

print(
    f"Failed requests:           "
    f"{failed_count}/{CONCURRENCY}"
)

print(
    f"Wall time:                 "
    f"{wall_time:.3f}s"
)

print(
    f"Request throughput:        "
    f"{request_throughput:.2f} req/s"
)


print()
print("Tokens:")

print(
    f"Total prompt tokens:       "
    f"{total_prompt_tokens:,}"
)

print(
    f"Total output tokens:       "
    f"{total_output_tokens:,}"
)

print(
    f"Aggregate output tok/s:    "
    f"{output_tok_s:.2f}"
)


print()
print("Latency:")

print(
    f"Mean:                       "
    f"{mean_latency:.3f}s"
)

print(
    f"P50:                        "
    f"{p50_latency:.3f}s"
)

print(
    f"P95:                        "
    f"{p95_latency:.3f}s"
)

print(
    f"P99:                        "
    f"{p99_latency:.3f}s"
)


print()
print("Scheduler / resource pressure:")

print(
    f"Max running:                "
    f"{max_running:.0f}"
)

print(
    f"Max waiting:                "
    f"{max_waiting:.0f}"
)

print(
    f"Max capacity waiting:       "
    f"{max_capacity_waiting:.0f}"
)

print(
    f"Max deferred waiting:       "
    f"{max_deferred_waiting:.0f}"
)

print(
    f"Peak KV cache usage:        "
    f"{peak_kv:.2f}%"
)

print(
    f"Metric samples:             "
    f"{len(samples)}"
)


# ============================================================
# Failed request diagnostics
# ============================================================

if failed:

    print()
    print("========================================")
    print("FIRST FAILED REQUEST")
    print("========================================")

    first_failure = failed[0]

    print(
        "Status:",
        first_failure["status_code"],
    )

    print(
        "Error:",
        first_failure["error"],
    )


# ============================================================
# Result JSON
# ============================================================

result = {

    "experiment":
        "phase_6_scheduler_resource_pressure",

    "configuration": {

        "model": MODEL,

        "concurrency":
            CONCURRENCY,

        "prompt_tokens":
            len(final_tokens),

        "max_new_tokens":
            MAX_NEW_TOKENS,

        "max_model_len":
            4096,

        "sample_interval":
            SAMPLE_INTERVAL,
    },

    "results": {

        "successful_requests":
            completed,

        "failed_requests":
            failed_count,

        "wall_time_seconds":
            wall_time,

        "request_throughput":
            request_throughput,

        "total_prompt_tokens":
            total_prompt_tokens,

        "total_output_tokens":
            total_output_tokens,

        "output_tokens_per_second":
            output_tok_s,

        "mean_latency_seconds":
            mean_latency,

        "p50_latency_seconds":
            p50_latency,

        "p95_latency_seconds":
            p95_latency,

        "p99_latency_seconds":
            p99_latency,
    },

    "scheduler_metrics": {

        "max_running":
            max_running,

        "max_waiting":
            max_waiting,

        "max_capacity_waiting":
            max_capacity_waiting,

        "max_deferred_waiting":
            max_deferred_waiting,

        "peak_kv_cache_usage_percent":
            peak_kv,
    },

    "metric_samples":
        samples,
}


# ============================================================
# Save JSON
# ============================================================

json_path = (
    OUTPUT_DIR /
    "scheduler_pressure.json"
)

with open(json_path, "w") as f:

    json.dump(
        result,
        f,
        indent=2,
    )


# ============================================================
# Save metrics CSV
# ============================================================

csv_path = (
    OUTPUT_DIR /
    "scheduler_metrics.csv"
)

with open(
    csv_path,
    "w",
    newline="",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "time",
            "running",
            "waiting",
            "capacity_waiting",
            "deferred_waiting",
            "kv_cache_usage",
        ],
    )

    writer.writeheader()

    writer.writerows(samples)


print()
print("========================================")
print("FILES SAVED")
print("========================================")

print(json_path)
print(csv_path)

Checking vLLM server...
{'running': 0, 'waiting': 0, 'kv_cache_usage': 0, 'capacity_waiting': 0, 'deferred_waiting': 0}

Tokenizing workload...
Candidate tokens: 49,601
Final prompt tokens: 3,000
Prompt characters:    17,169

Running warm-up...
Warm-up complete.

PHASE 6 RESOURCE PRESSURE EXPERIMENT
Model:                Qwen/Qwen2.5-1.5B-Instruct
Concurrency:          128
Prompt tokens:        3,000
Max new tokens:       100
Total token/request:  3,100
Sampling interval:    0.05s

Starting workload...

Completed: 16/128
Completed: 32/128
Completed: 48/128
Completed: 64/128
Completed: 80/128
Completed: 96/128
Completed: 112/128
Completed: 128/128

RESULTS

Requests:
Successful requests:       128/128
Failed requests:           0/128
Wall time:                 32.341s
Request throughput:        3.96 req/s

Tokens:
Total prompt tokens:       384,000
Total output tokens:       12,800
Aggregate output tok/s:    395.79

Latency:
Mean:                       30.901s
P50:                      

In [ ]:
import requests

text = requests.get("http://127.0.0.1:8000/metrics").text

for line in text.splitlines():
    if "vllm:" in line and (
        "request" in line
        or "kv_cache" in line
        or "waiting" in line
    ):
        print(line)

# HELP vllm:num_requests_running Number of requests in model execution batches.
# TYPE vllm:num_requests_running gauge
vllm:num_requests_running{engine="0",model_name="Qwen/Qwen2.5-1.5B-Instruct"} 0.0
# HELP vllm:num_requests_waiting Number of requests waiting to be processed.
# TYPE vllm:num_requests_waiting gauge
vllm:num_requests_waiting{engine="0",model_name="Qwen/Qwen2.5-1.5B-Instruct"} 0.0
# HELP vllm:num_requests_waiting_by_reason Number of waiting requests by reason. Reason labels: 'capacity' = waiting for scheduling capacity; 'deferred' = deferred by transient constraints (LoRA budget, KV transfer, blocked status). Sum of all reasons equals vllm:num_requests_waiting.
# TYPE vllm:num_requests_waiting_by_reason gauge
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-1.5B-Instruct",reason="capacity"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-1.5B-Instruct",reason="deferred"} 0.0
# HELP vllm:kv_cache_usage_perc KV-cache usage

In [ ]:
print("\n--- ALL RELEVANT METRICS ---")

for line in text.splitlines():
    if any(x in line.lower() for x in [
        "vllm:num_requests",
        "vllm:kv_cache",
        "vllm:request",
    ]):
        print(line)


--- ALL RELEVANT METRICS ---
# HELP vllm:num_requests_running Number of requests in model execution batches.
# TYPE vllm:num_requests_running gauge
vllm:num_requests_running{engine="0",model_name="Qwen/Qwen2.5-1.5B-Instruct"} 0.0
# HELP vllm:num_requests_waiting Number of requests waiting to be processed.
# TYPE vllm:num_requests_waiting gauge
vllm:num_requests_waiting{engine="0",model_name="Qwen/Qwen2.5-1.5B-Instruct"} 0.0
# HELP vllm:num_requests_waiting_by_reason Number of waiting requests by reason. Reason labels: 'capacity' = waiting for scheduling capacity; 'deferred' = deferred by transient constraints (LoRA budget, KV transfer, blocked status). Sum of all reasons equals vllm:num_requests_waiting.
# TYPE vllm:num_requests_waiting_by_reason gauge
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-1.5B-Instruct",reason="capacity"} 0.0
vllm:num_requests_waiting_by_reason{engine="0",model_name="Qwen/Qwen2.5-1.5B-Instruct",reason="deferred"} 0.0
# HELP vllm:kv_c

In [ ]:
!python /content/phase6_scheduler.py

python3: can't open file '/content/phase6_scheduler.py': [Errno 2] No such file or directory


In [ ]:
import requests

r = requests.get("http://127.0.0.1:8000/metrics")
print(r.status_code)

200


In [ ]:
!git clone https://github.com/chistym17/llm-inference-lab.git

Cloning into 'llm-inference-lab'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 63 (delta 10), reused 59 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 16.63 KiB | 4.16 MiB/s, done.
Resolving deltas: 100% (10/10), done.


In [ ]:
%cd llm-inference-lab

/content/llm-inference-lab


In [ ]:
!find experiments/04_concurrency src/llm_platform/benchmark -maxdepth 2 -type f | sort

experiments/04_concurrency/README.md
experiments/04_concurrency/run.py
src/llm_platform/benchmark/concurrency.py
src/llm_platform/benchmark/metrics.py
src/llm_platform/benchmark/results.py
src/llm_platform/benchmark/runner.py
src/llm_platform/benchmark/workloads.py


In [ ]:
!pip install -e .

Obtaining file:///content/llm-inference-lab
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for open-llm-inference-platform (pyproject.toml) ... done
  Created wheel for open-llm-inference-platform: filename=open_llm_inference_platform-0.1.0-0.editable-py3-none-any.whl size=1509 sha256=1b85ef4a11177f2a08e266e5d6d197c7fffa022ab442041c5433e72391c262d0
  Stored in directory: /tmp/pip-ephem-wheel-cache-l1z6vpdx/wheels/91/54/35/9266268efc9c6a7bb2a580ff53435ec52310b1145d70948c0e
Successfully built open-llm-inference-platform


In [ ]:
!PYTHONPATH=src python experiments/04_concurrency/run.py

/usr/local/lib/python3.13/dist-packages/torch/cuda/__init__.py:64: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Checking vLLM server...
KV cache: 0.0%
Running:  0.0
Waiting:  0.0

=== Concurrency 1 ===
Completed:          1
Wall time:          3.188s
Requests/sec:       0.31
Aggregate tok/s:    31.37
Mean latency:       3.187s
P50 latency:        3.187s
P95 latency:        3.187s
KV cache after:     0.0%

=== Concurrency 2 ===
Completed:          2
Wall time:          1.517s
Requests/sec:       1.32
Aggregate tok/s:    131.82
Mean latency:       1.508s
P50 latency:        1.508s
P95 latency:        1.512s
KV cache after:     0.0%

=== Concurrency 4 ===
Completed:          4
Wall time:          1.526s
Requests/sec:       2.62
Aggregate tok/s:    262.08
Mean latency:       1.519s
P50 la